# Lab 3.1 — Model Serialization & Format Comparison
**Module 3: Bridging ML and Engineering**

In this lab you will:
- Compare three serialization formats: **joblib**, **pickle**, and **ONNX**
- Measure file size and inference latency for each format
- Export a trained scikit-learn pipeline to ONNX and run it with `onnxruntime`
- Verify prediction consistency across all formats
- Understand when to use each format in production

> **Instructor Note:** Serialization is the bridge between model training (Python) and model serving (any language, any platform). ONNX is the key for cross-platform deployment — an XGBoost model trained in Python can run in a .NET or Java service via ONNX Runtime.


## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| scikit-learn | `scikit-learn` |
| xgboost | `xgboost` |
| joblib | `joblib` |
| skl2onnx | `skl2onnx` |
| onnxruntime | `onnxruntime` |
| pandas | `pandas` |
| numpy | `numpy` |

**Install all at once:**
```bash
pip install scikit-learn xgboost joblib skl2onnx onnxruntime pandas numpy
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


In [1]:
import subprocess, sys

required = {
    'joblib': 'joblib',
    'sklearn': 'scikit-learn',
    'skl2onnx': 'skl2onnx',
    'onnxruntime': 'onnxruntime',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'xgboost': 'xgboost',
}
for pkg, inst in required.items():
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', inst, '--quiet', '--break-system-packages'])
print("All packages ready ✅")


All packages ready ✅


## 1. Load the Trained Model and Data

We load the best_tuned_model from Module 2. If it isn't found, we train a fresh RandomForest on synthetic Nutanix telemetry as a fallback.

> **Instructor Note:** In production you always version your models. The `_v1` suffix in the filename is intentional — it signals model versioning. Lab 3.2 will build a service that serves this model.


In [2]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib, pickle

warnings.filterwarnings('ignore')

MODEL_PATH = os.path.join('..', 'Module_2', 'best_tuned_model.pkl')
DATA_PATH  = os.path.join('..', 'Module_1', 'features_engineered.csv')

# ── Load or create a model ─────────────────────────────────────────────────
try:
    model = joblib.load(MODEL_PATH)
    MODEL_SOURCE = 'Module 2 tuned model'
    print(f"✅ Loaded model from {MODEL_PATH}")
except FileNotFoundError:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    np.random.seed(42)
    n = 800
    X_syn = pd.DataFrame({
        'cpu_percent': np.random.uniform(5, 100, n),
        'memory_usage_gb': np.random.uniform(4, 64, n),
        'disk_io_mbps': np.random.uniform(10, 500, n),
        'network_rx_mbps': np.random.uniform(1, 200, n),
        'network_tx_mbps': np.random.uniform(1, 200, n),
        'active_vms': np.random.randint(1, 40, n),
        'stargate_ops': np.random.randint(100, 5000, n),
        'cerebro_replication_lag_s': np.random.uniform(0, 120, n),
    })
    y_syn = ((X_syn['cpu_percent'] > 80) | (X_syn['memory_usage_gb'] > 55)).astype(int)
    model = Pipeline([('scaler', StandardScaler()),
                      ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
    model.fit(X_syn, y_syn)
    MODEL_SOURCE = 'synthetic fallback model'
    DATA_PATH = None
    print("⚠️  Module 2 model not found — using synthetic fallback model")

print(f"Model source: {MODEL_SOURCE}")
print(f"Model type: {type(model).__name__}")


✅ Loaded model from ../Module_2/best_tuned_model.pkl
Model source: Module 2 tuned model
Model type: XGBClassifier


## 2. Load Evaluation Data

We use the same features_engineered.csv from Module 1 to create a consistent test set for comparing predictions across serialization formats.


In [3]:
if DATA_PATH and os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    DROP = [c for c in ['is_high_cpu', 'is_slow_response', 'message', 'date', 'is_anomaly'] if c in df.columns]
    X_eval = df.drop(columns=DROP).dropna()
    y_eval = df.loc[X_eval.index, 'is_high_cpu'].astype(int) if 'is_high_cpu' in df.columns else pd.Series(np.zeros(len(X_eval)))
    DATA_SOURCE = 'Module 1 CSV'
else:
    np.random.seed(99)
    n = 460
    ops = ['READ', 'WRITE', 'DELETE', 'UPDATE']
    X_eval = pd.DataFrame({
        'cpu_percent': np.random.uniform(5, 100, n),
        'memory_mb': np.random.uniform(4096, 65536, n),
        'disk_io_mbps': np.random.uniform(10, 500, n),
        'response_time_ms': np.random.uniform(1, 2000, n),
        'hour_of_day': np.random.randint(0, 24, n),
        'day_of_week': np.random.randint(0, 7, n),
        'operation_type': np.random.choice(ops, n),
        'is_weekend': np.random.randint(0, 2, n),
        'is_business_hours': np.random.randint(0, 2, n),
        'time_since_last_error_s': np.random.uniform(0, 3600, n),
        'cpu_rolling_mean_5': np.random.uniform(5, 100, n),
        'memory_rolling_mean_5': np.random.uniform(4096, 65536, n),
        'error_rate_per_host': np.random.randint(0, 20, n),
        'component_error_count': np.random.randint(0, 50, n),
    })
    y_eval = pd.Series(np.zeros(n))
    DATA_SOURCE = 'synthetic eval data'

# Align to exact features the model was trained on (adds missing cols with 0)
try:
    expected = model.feature_names_in_.tolist()
    for col in expected:
        if col not in X_eval.columns:
            X_eval[col] = 0
    X_eval = X_eval[expected]
except AttributeError:
    pass  # fallback model has no feature_names_in_

string_cols = X_eval.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Data source   : {DATA_SOURCE}")
print(f"Eval set      : {X_eval.shape[0]} rows × {X_eval.shape[1]} features")
print(f"Categorical   : {string_cols}")


Data source   : Module 1 CSV
Eval set      : 413 rows × 33 features
Categorical   : ['operation_type']


## 3. Serialization Format 1 — joblib

`joblib` is the standard for scikit-learn models. It uses memory-mapped arrays and is faster than pickle for large numpy arrays embedded in models (e.g., Random Forest trees).

> **Instructor Note:** joblib uses numpy memory-mapping. For a 100-tree Random Forest, it can be 2–3× faster to load than pickle because the tree arrays are mmap'd from disk rather than deserialized.


In [4]:
SAVE_DIR = '.'  # save in Module_3 directory
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save with joblib ──────────────────────────────────────────────────────
joblib_path = os.path.join(SAVE_DIR, 'model_joblib.pkl')
t0 = time.perf_counter()
joblib.dump(model, joblib_path, compress=3)  # compress=3 is a good balance
joblib_save_time = time.perf_counter() - t0

# ── Load with joblib ──────────────────────────────────────────────────────
t0 = time.perf_counter()
model_jl = joblib.load(joblib_path)
joblib_load_time = time.perf_counter() - t0

# ── Inference time ────────────────────────────────────────────────────────
# 1. Convert the string column to a pandas 'category' datatype
X_eval['operation_type'] = X_eval['operation_type'].astype('category')

# 2. Run inference time code as normal
t0 = time.perf_counter()
preds_joblib = model_jl.predict(X_eval)
joblib_infer_time = (time.perf_counter() - t0) * 1000  # ms

# THIS LINE WAS MISSING: Calculate the file size
joblib_size_kb = os.path.getsize(joblib_path) / 1024

print(f"joblib (compress=3):")
print(f"  Save time  : {joblib_save_time*1000:.1f} ms")
print(f"  Load time  : {joblib_load_time*1000:.1f} ms")
print(f"  Infer time : {joblib_infer_time:.2f} ms  ({X_eval.shape[0]} rows)")
print(f"  File size  : {joblib_size_kb:.1f} KB")
print(f"  Predictions: {preds_joblib[:10].tolist()}")

joblib (compress=3):
  Save time  : 1.9 ms
  Load time  : 1.1 ms
  Infer time : 2.27 ms  (413 rows)
  File size  : 13.2 KB
  Predictions: [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]


## 4. Serialization Format 2 — pickle

`pickle` is Python's built-in serialization. It works for any Python object but has no compression and no language portability — a pickled model can only be loaded in Python.

> **Instructor Note:** Never use pickle for untrusted data — loading a pickle file executes arbitrary code. In production, only load pickle/joblib files from your own trusted artifact registry.


In [5]:
# ── Save with pickle ─────────────────────────────────────────────────────
pickle_path = os.path.join(SAVE_DIR, 'model_pickle.pkl')
t0 = time.perf_counter()
with open(pickle_path, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
pickle_save_time = time.perf_counter() - t0

# ── Load with pickle ──────────────────────────────────────────────────────
t0 = time.perf_counter()
with open(pickle_path, 'rb') as f:
    model_pk = pickle.load(f)
pickle_load_time = time.perf_counter() - t0

# ── Inference time ────────────────────────────────────────────────────────
t0 = time.perf_counter()
preds_pickle = model_pk.predict(X_eval)
pickle_infer_time = (time.perf_counter() - t0) * 1000

pickle_size_kb = os.path.getsize(pickle_path) / 1024

print(f"pickle (HIGHEST_PROTOCOL):")
print(f"  Save time  : {pickle_save_time*1000:.1f} ms")
print(f"  Load time  : {pickle_load_time*1000:.1f} ms")
print(f"  Infer time : {pickle_infer_time:.2f} ms  ({X_eval.shape[0]} rows)")
print(f"  File size  : {pickle_size_kb:.1f} KB")
print(f"  Predictions: {preds_pickle[:10].tolist()}")

# Verify consistency
assert np.array_equal(preds_joblib, preds_pickle), "❌ joblib vs pickle mismatch!"
print("\n✅ joblib and pickle produce identical predictions")


pickle (HIGHEST_PROTOCOL):
  Save time  : 1.5 ms
  Load time  : 0.9 ms
  Infer time : 1.32 ms  (413 rows)
  File size  : 204.4 KB
  Predictions: [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]

✅ joblib and pickle produce identical predictions


## 5. Serialization Format 3 — ONNX

ONNX (Open Neural Network Exchange) is a cross-platform format. A model exported to ONNX can run in C++, C#, Java, and any environment with `onnxruntime` — no Python, no scikit-learn dependency required.

> **Instructor Note:** ONNX requires all inputs to be numeric. Our model has a string categorical feature (`operation_type`). This is a real-world ONNX deployment challenge — we label-encode it before conversion and note the encoding map in the model card so the serving layer can reproduce it. This pattern is standard: preprocessing lives outside the ONNX graph when the input layer can't represent it natively.

In [6]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
from sklearn.preprocessing import LabelEncoder
import onnxruntime as rt
import numpy as np

# ── Step 1: Label-encode string columns (ONNX requires all-numeric input) ──
X_onnx = X_eval.copy()
label_encoders = {}
for col in string_cols:
    le = LabelEncoder()
    X_onnx[col] = le.fit_transform(X_onnx[col].astype(str))
    label_encoders[col] = {cls: int(idx) for idx, cls in enumerate(le.classes_)}
    print(f"  Encoded '{col}': {label_encoders[col]}")

n_features_onnx = X_onnx.shape[1]
X_float = X_onnx.values.astype(np.float32)
print(f"\nONNX input: {n_features_onnx} numeric features")

# ── Step 2: Train a numeric-only XGBoost for export ──────────────────────
# The loaded model uses string categoricals which ONNX can't represent
# natively, so we retrain with the same hyperparameters on encoded data.
from xgboost import XGBClassifier

xgb_for_onnx = XGBClassifier(
    n_estimators=model.n_estimators,
    max_depth=model.max_depth,
    learning_rate=model.learning_rate,
    random_state=42,
    eval_metric='logloss',
)
y_onnx = y_eval.values if hasattr(y_eval, 'values') else np.zeros(len(X_onnx))
xgb_for_onnx.fit(X_float, y_onnx)

# ── Step 3: Convert to ONNX ───────────────────────────────────────────────
initial_type = [('float_input', FloatTensorType([None, n_features_onnx]))]
onnx_path = os.path.join(SAVE_DIR, 'model.onnx')

t0 = time.perf_counter()
onnx_model = convert_xgboost(xgb_for_onnx, initial_types=initial_type, target_opset=15)
onnx_save_time = time.perf_counter() - t0

with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

onnx_size_kb = os.path.getsize(onnx_path) / 1024
print(f"\nONNX export:")
print(f"  Conversion time : {onnx_save_time*1000:.1f} ms")
print(f"  File size       : {onnx_size_kb:.1f} KB")

# ── Step 4: Run inference with onnxruntime ────────────────────────────────
sess = rt.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name   # 'label'
prob_name   = sess.get_outputs()[1].name   # 'probabilities'

print(f"\nONNX session:")
print(f"  Input  : '{input_name}' {sess.get_inputs()[0].shape}")
print(f"  Outputs: {[o.name for o in sess.get_outputs()]}")

t0 = time.perf_counter()
preds_onnx = sess.run([output_name], {input_name: X_float})[0]
onnx_infer_time = (time.perf_counter() - t0) * 1000

print(f"\nONNX inference:")
print(f"  Infer time  : {onnx_infer_time:.2f} ms  ({X_float.shape[0]} rows)")
print(f"  Predictions : {preds_onnx[:10].tolist()}")
print("\n⚠️  ONNX predictions differ from joblib/pickle: this model was retrained")
print("    on label-encoded features. In production, apply the same encoding")
print("    before calling the ONNX runtime.")

  Encoded 'operation_type': {'DELETE': 0, 'READ': 1, 'UPDATE': 2, 'WRITE': 3}

ONNX input: 33 numeric features

ONNX export:
  Conversion time : 5.4 ms
  File size       : 12.9 KB

ONNX session:
  Input  : 'float_input' [None, 33]
  Outputs: ['label', 'probabilities']

ONNX inference:
  Infer time  : 0.81 ms  (413 rows)
  Predictions : [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]

⚠️  ONNX predictions differ from joblib/pickle: this model was retrained
    on label-encoded features. In production, apply the same encoding
    before calling the ONNX runtime.


## 6. Format Comparison Summary

> **Instructor Note:** Walk through the table. Key message: joblib = Python-only production; ONNX = cross-platform production. Pickle is fine for notebooks and experiments but not for services.


In [7]:
_onnx_kb    = round(onnx_size_kb, 1)    if 'onnx_size_kb'    in dir() else 'N/A'
_onnx_infer = round(onnx_infer_time, 2) if 'onnx_infer_time' in dir() else 'N/A'

comparison = pd.DataFrame({
    'Format':          ['joblib (compress=3)', 'pickle', 'ONNX'],
    'Size (KB)':       [round(joblib_size_kb, 1), round(pickle_size_kb, 1), _onnx_kb],
    'Load (ms)':       [round(joblib_load_time*1000, 1), round(pickle_load_time*1000, 1), 'N/A (session)'],
    'Infer (ms)':      [round(joblib_infer_time, 2), round(pickle_infer_time, 2), _onnx_infer],
    'Cross-language':  ['No', 'No', 'Yes'],
    'Compression':     ['Built-in (zlib)', 'None', 'N/A'],
    'Best for': [
        'Python prod services',
        'Experiments / notebooks',
        'Multi-language / edge deploys',
    ]
})
print(comparison.to_string(index=False))

             Format  Size (KB)     Load (ms)  Infer (ms) Cross-language     Compression                      Best for
joblib (compress=3)       13.2           1.1        2.27             No Built-in (zlib)          Python prod services
             pickle      204.4           0.9        1.32             No            None       Experiments / notebooks
               ONNX       12.9 N/A (session)        0.81            Yes             N/A Multi-language / edge deploys


## 7. Model Metadata File

> **Instructor Note:** A serialized model file is useless without metadata — what features it expects, what version it is, when it was trained. We write a `model_card.json` file alongside the model. This is standard practice in production ML.


In [8]:
metadata = {
    'model_name': 'nutanix_anomaly_detector',
    'version': '1.0.0',
    'source': MODEL_SOURCE,
    'features': list(X_eval.columns),
    'n_features': X_eval.shape[1],
    'categorical_encodings': label_encoders if 'label_encoders' in dir() else {},
    'target': 'is_anomaly (1=anomalous CVM, 0=normal)',
    'serialization_formats': {
        'joblib': 'model_joblib.pkl',
        'pickle': 'model_pickle.pkl',
        'onnx':   'model.onnx (numeric-only, requires label encoding)',
    },
    'inference_latency_ms': {
        'joblib': round(joblib_infer_time, 2),
        'pickle': round(pickle_infer_time, 2),
        'onnx':   round(onnx_infer_time, 2) if 'onnx_infer_time' in dir() else None,
    },
    'file_sizes_kb': {
        'joblib': round(joblib_size_kb, 1),
        'pickle': round(pickle_size_kb, 1),
        'onnx':   round(onnx_size_kb, 1)  if 'onnx_size_kb'    in dir() else None,
    }
}

card_path = os.path.join(SAVE_DIR, 'model_card_v2.json')
with open(card_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))
print(f"\n✅ model_card_v2.json written to {card_path}")

{
  "model_name": "nutanix_anomaly_detector",
  "version": "1.0.0",
  "source": "Module 2 tuned model",
  "features": [
    "cpu_percent",
    "memory_mb",
    "disk_io_mbps",
    "response_time_ms",
    "hour_of_day",
    "day_of_week",
    "operation_type",
    "is_weekend",
    "is_business_hours",
    "time_since_last_error_s",
    "cpu_rolling_mean_5",
    "memory_rolling_mean_5",
    "error_rate_per_host",
    "component_error_count",
    "loglevel_CRITICAL",
    "loglevel_DEBUG",
    "loglevel_ERROR",
    "loglevel_INFO",
    "loglevel_WARNING",
    "comp_AOS",
    "comp_Acropolis",
    "comp_Cerebro",
    "comp_Curator",
    "comp_Prism",
    "comp_Stargate",
    "host_encoded",
    "cpu_percent_robust",
    "memory_mb_robust",
    "disk_io_mbps_robust",
    "response_time_ms_robust",
    "log_response_time",
    "cpu_memory_ratio",
    "io_per_cpu"
  ],
  "n_features": 33,
  "categorical_encodings": {
    "operation_type": {
      "DELETE": 0,
      "READ": 1,
      "UPDATE": 

## 8. Lab Summary

| Format | Size | Speed | Portability | Use case |
|--------|------|-------|-------------|----------|
| joblib | Compressed | Fast load | Python only | Python microservices |
| pickle | Uncompressed | Medium | Python only | Notebooks, scripts |
| ONNX   | Small graph | Fast infer | Any language | Edge, multi-language |

**Files created in `Module_3/`:**
- `model_joblib.pkl` — compressed joblib model
- `model_pickle.pkl` — pickle model  
- `model.onnx` — ONNX graph
- `model_card_v2.json` — model metadata

> **Instructor Note:** In the next lab, Lab 3.2, we'll wrap `model_joblib.pkl` in a FastAPI service and expose it as a REST endpoint.

---
## 🎯 Challenges

### Challenge 1
Load `model_joblib.pkl` and `model.onnx`. Run both on a **manually crafted input** representing a CVM with `cpu_percent=92`, `memory_usage_gb=60`, `disk_io_mbps=450` (fill remaining features with their mean from `X_eval`). Do both formats agree on whether it's anomalous?


In [9]:
# Challenge 1 — your code here


### Challenge 2
Re-save the model using joblib with `compress=0` (no compression) and `compress=9` (max compression). Compare file sizes and load times across all three compression levels. Which level offers the best size/speed trade-off?


In [10]:
# Challenge 2 — your code here


### Challenge 3
The ONNX model outputs class labels. Modify the inference code to also extract **class probabilities** from the ONNX session (hint: check `sess.get_outputs()` — there should be a `probabilities` output). Print the probability of anomaly for the first 5 rows.


In [11]:
# Challenge 3 — your code here
